# R2 pipeline — plots (thin driver)

Spec §4.8: one parameter cell (`RUN_DIR`), then one cell per `src/rngrn/viz.py` figure
function. Every figure is drawn from `RUN_DIR/arrays/plot_arrays.npz`
(`rngrn.plotdata.load_plot_arrays`) — the persisted plottable arrays a run writes
(`train.py::_save_run_arrays`), never `payload.h5` and never an answer key. This notebook
defines no helper functions; all plotting logic lives in `viz.py`.

**Not executed as committed.** `RUN_DIR` below points at the Phase-I run directory the
pipeline notebook launches, but no run has produced it yet — `redesign_pipeline.ipynb`'s
launch cell has never been run (Task 16's `scripts/r2_ignition_run.py` does not exist yet).
Point `RUN_DIR` at any completed run's directory to use this notebook today.

In [ ]:
import json
import os
import sys

REPO = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks"
        else os.path.abspath(os.getcwd()))
sys.path.insert(0, os.path.join(REPO, "src"))

RUN_DIR = os.path.join(REPO, "experiments", "redesign_r2", "phase1")  # <-- point at a run

## Load the run's plottable arrays

In [ ]:
from rngrn.plotdata import load_plot_arrays, plot_arrays_path

arrays_path = plot_arrays_path(RUN_DIR)
arrays = load_plot_arrays(arrays_path)
print(f"loaded {len(arrays)} array(s) from {arrays_path}")
print(json.dumps(arrays.get("meta", {}), indent=2, default=str))

## Loss curves

In [ ]:
from rngrn import viz

FIG_DIR = os.path.join(RUN_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

viz.loss_curves(arrays, os.path.join(FIG_DIR, "loss_curves.png"))

## Invariant trajectories

In [ ]:
viz.invariant_trajectories(arrays, os.path.join(FIG_DIR, "invariant_trajectories.png"))

## Population events

In [ ]:
viz.event_timeline(arrays, os.path.join(FIG_DIR, "event_timeline.png"))

## Spectral trace

`SPEC_SHAPE_FLOOR` is D3's measured patch-to-patch estimation floor for the spec_shape-form
log-RAPS distance (`docs/DIAGNOSTICS_fft.md`, closure of D-FFT-9): mean 0.389 on
`[0.5, 1.5]·k*` (range 0.122-0.621), i.e. ~31%/bin power variation patch-to-patch. A fit
error drawn below this reference line is fitting estimation noise, not signal — it is a
measured number, not chosen here.

In [ ]:
SPEC_SHAPE_FLOOR = 0.389  # docs/DIAGNOSTICS_fft.md D3 closure (D-FFT-9); measured, not tuned

viz.spectral_trace(arrays, SPEC_SHAPE_FLOOR, os.path.join(FIG_DIR, "spectral_trace.png"))